In [1]:
# Python
import pickle
import sys
import time
from datetime import datetime
import pytz
# Python Files
from satellite_RFI.src import check_satellite as cs
from satellite_RFI.src import beam_model as bm
from satellite_RFI.src import rewrite_tle
# Astropy
from astropy.time import Time
import astropy.units as u
import astropy.constants as cc
# Numpy
import numpy as np
#  Matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker


In [53]:
katdal_info = pickle.load(open('/idia/projects/hi_im/satellite_rfi/high_frequency/1551055211_level6_mask/1551055211_katdal_info.p', 'rb'))
info = [katdal_info[i] for i in katdal_info.keys()]
obs_time_input='2019 2 25 0 40 11'
tle_location='TLE/2019_02_21_tle/'

nd_s0=info[1]
nd_s0_pos=None
nd_s=None
nd_s0_coords=info[5]
frequency=info[2]
fs=1000
fe=1400
timestamps=None
telescope_Lon =  (21. + 26./60. + 38.00/3600.) * u.deg #
telescope_Lat = -(30. + 42./60. + 47.41/3600.) * u.deg #
data_save='test/'

 
sats_type =['gps-ops', 'glo-ops', 'galileo', 'beidou', 'irnss', 'qzs', 'sbas']   
tle_data_sort=1

In [84]:
class satellite_positions(object):
    def __init__(self,
                observation_start=None,
                tle_location=None,
                sat_location=None, # Must still do
                timesteps=None,
                pointing_coords=None):
        
        self.obs_start=observation_start
        self.tle_loc=tle_location
        self.sats_constellation=['gps-ops', 'glo-ops', 'galileo', 'beidou', 'irnss', 'qzs', 'sbas']
        self.timesteps=timesteps
        self.RA, self.DEC=pointing_coords
        
        
        
        
        self.tele_Long, self.tele_Lat=(21. + 26./60. + 38.00/3600.) * u.deg, -(30. + 42./60. + 47.41/3600.) * u.deg # fix later
        
    def file_name(self):
        """
        Establishing the file name
        """
        obs_start=[int(x) for x in self.obs_start.split()]
        observation_s = datetime(obs_start[0], obs_start[1], obs_start[2], obs_start[3], obs_start[4], obs_start[5])
        fname = int((observation_s - datetime(1970, 1, 1)).total_seconds())
#         print "File name to be used is: ", fname
        return fname
    
    def rewrite_TLE(self):
        """
        Rewriting the TLE data
        """
        if tle_data_sort is not None: 
            print 'Re-writing data'
            rewrite_tle.rewrite_sat_cat(file_path=tle_location)
            print 'Re-write completed'
            

            
    # Calling in another class function from check_satellite.py
    def satellite_data(self):
        tsc = cs.Telescopesite_Satellite_Catalogue(source_url=self.tle_loc, 
                                                   sats_type=self.sats_constellation, reload=False)

        tsc.obs_time= datetime.utcfromtimestamp(int(self.file_name())+self.timesteps[0]).strftime('%Y-%m-%d %H:%M:%S')
        tsc.obs_time_list = (self.timesteps - self.timesteps[0]) * u.second
        tsc.obs_location = [self.tele_Long, self.tele_Lat]

        tsc.get_sate_coords()


        telescope_pointing=np.array([[self.RA[i], self.DEC[i]] for i in range(len(self.RA))])
        
    def satellite_angle(self):
        satellite_angle = tsc.check_angular_separation(pointings=telescope_pointing, max_angle=100, beam_func=None, 
            ymin=1e-10, ymax=1, axes=None)
        
        return satellite_angle

In [85]:
xx = satellite_positions(observation_start='2019 2 25 0 40 11', 
                   tle_location='TLE/2019_02_21_tle/', timesteps=info[1], pointing_coords=info[5])

In [83]:
xx.satellite_data()

load sat catalogue from TLE/2019_02_21_tle/ gps-ops.txt
load sat catalogue from TLE/2019_02_21_tle/ glo-ops.txt
load sat catalogue from TLE/2019_02_21_tle/ galileo.txt
load sat catalogue from TLE/2019_02_21_tle/ beidou.txt
load sat catalogue from TLE/2019_02_21_tle/ irnss.txt
load sat catalogue from TLE/2019_02_21_tle/ qzs.txt
load sat catalogue from TLE/2019_02_21_tle/ sbas.txt
Time range 2019-02-25 00:53:05.000 - 2019-02-25 02:28:04.589
Satellite      gps-ops has   17 satellites  [use  10.02 s]
Satellite      glo-ops has   14 satellites  [use   8.40 s]
Satellite      galileo has   12 satellites  [use   7.31 s]
Satellite       beidou has   12 satellites  [use   7.38 s]
Satellite        irnss has    3 satellites  [use   2.75 s]
Satellite          qzs has    0 satellites [Removing this constellation from sats_type]
Satellite         sbas has    5 satellites  [use   3.99 s]


In [86]:
xx.satellite_angle()

NameError: global name 'tsc' is not defined

In [ ]:
class Telescopesite_Satellite_Catalogue(Satellite_Catalogue):

    #def __init__(self, reload=False, source_url=None):
    def __init__(self, *args, **kwargs):

        super(Telescopesite_Satellite_Catalogue, self).__init__(*args, **kwargs)
        
#         MeerKAT information
#         telescope_Lon =  (21. + 26./60. + 38.00/3600.) * u.deg #
#         telescope_Lat = -(30. + 42./60. + 47.41/3600.) * u.deg #
#         self.obs_location = [telescope_Lat, telescope_Lon]

    def get_angular_separation(self, pointings, beam_func=None):

        """Obsolete, beam function are given in the beam function package"""
#         if beam_func is None:
#             # using modified sinc function, quite close to Khan's model
#             beam_func = lambda x: np.sinc(x) ** 2 * ((1/(np.abs(x) + 1)) ** 2.5)

        super(Telescopesite_Satellite_Catalogue, self).get_angular_separation(
                pointings, beam_func=beam_func)

    def check_angular_separation(self, pointings, max_angle=None, beam_func=None,
            ymin=None, ymax=None, axes=None):
        

        """Obsolete, beam function are given in the beam function package"""
#         if beam_func is None:
#             # using modified sinc function, quite close to Khan's model
#             beam_func = lambda x: np.sinc(x) ** 2 * ((1/(np.abs(x) + 1)) ** 2.5)
         
        return super(Telescopesite_Satellite_Catalogue, self).check_angular_separation(
            pointings, max_angle=max_angle,
            beam_func=beam_func, ymin=ymin, ymax=ymax, axes=axes)



    def check_pointing(self, altaz, figaxes=None, pad=2.):
        '''
        rising_altaz = [Az, Alt]
        setting_altaz = [Az, Alt]
        '''
        #_az0, _alt0 = altaz
        _az_min, _alt_min = np.min(altaz, axis=0)
        _az_max, _alt_max = np.max(altaz, axis=0)
        if _az_min < 0: _az_min += 360.
        if _az_max < 0: _az_max += 360.

        if figaxes is None:
            fig2 = plt.figure(figsize=(12, 12))
            gs = gridspec.GridSpec(1, 1, right=0.95, wspace=0.05, figure=fig2)
            ax_r = fig2.add_subplot(gs[0,0])
        else:
            fig2, ax_r = figaxes

        coord_list_1 = self.coord_list[0]
        name_list_1  = self.name_list[0]

        _sats_list1 = []

        legend_list = []
        for ii in range(len(self.sats_type)):
            coords_1 = coord_list_1[ii]
            coords_1 = np.array(coords_1)

            names_1 = name_list_1[ii]

            for jj in range(coords_1.shape[1]):
                good  = coords_1[:, jj, 1] > 0
                az  = coords_1[good, jj, 0] * 180./np.pi
                alt = coords_1[good, jj, 1] * 180./np.pi
                #az[az>180.] -= 360.
                ax_r.plot(az, alt, '.-', color=_c_list[ii], mfc=_c_list[ii],
                    mec='none', lw=0.1, ) #label=names_1[jj])

                good  = coords_1[:, jj, 0]* 180./np.pi > _az_min - pad
                good *= coords_1[:, jj, 0]* 180./np.pi < _az_max + pad
                good *= coords_1[:, jj, 1]* 180./np.pi > _alt_min - pad - 0.5
                good *= coords_1[:, jj, 1]* 180./np.pi < _alt_max + pad + 0.5
                if np.any(good):
                    az  = np.mean(coords_1[good, jj, 0] * 180./np.pi)
                    alt = np.mean(coords_1[good, jj, 1] * 180./np.pi)
                    _sats_list1.append([az, alt, r'%s'%names_1[jj], _c_list[ii]])

            legend_list.append(mpatches.Patch(color=_c_list[ii], 
                label='%s'%self.sats_type[ii]))
        #shift = 0.015
        shift = 0.02
        _sats_list1.sort( key=lambda x: x[1])
        for ss in range(len(_sats_list1)):
            _sats = _sats_list1[ss]
            ax_r.annotate(_sats[2].replace('&', r'\&'), (_sats[0], _sats[1]), 
                    textcoords = 'axes fraction',
                    xytext = (0.8, 0.02 + ss * shift),
                    arrowprops=dict(edgecolor=_sats[3],
                        arrowstyle='->', 
                        connectionstyle="arc,angleA=180,armA=100,rad=20"),
                    horizontalalignment='left', 
                    verticalalignment='bottom',
                    size = 8,
                    color=_sats[3])

        ax_r.legend(handles=legend_list, frameon=False, markerfirst=False,
                loc=1, )

        ax_r.set_title(self.obs_time[0])

        xx0 = np.linspace(_az_min, _az_max, 100)
        yy0 = np.ones_like(xx0) * (_alt_min + _alt_max) * 0.5
        ax_r.fill_between(xx0, yy0+0.5, yy0-0.5, facecolor='none', edgecolor='r')

        ax_r.set_xlim( _az_min - 10.,  _az_max + 10.)
        ax_r.set_ylabel(r'${\rm Alt}\, [^{\circ}]$')
        ax_r.set_xlabel(r'${\rm AZ}\, [^{\circ}]$')
        ax_r.tick_params(which='both', direction='in')

        _alt = (_alt_min + _alt_max) * 0.5
        ax_r.set_ylim(_alt - 5., _alt + 5.)
        plt.show()
        plt.close()
